# 5.01 Random Forest and Gradient Boosting Decision Trees

Completed adaptation of the [Warwick lecturer notebook](https://github.com/MJMortensonWarwick/IB2AD0_Data_Science_GenerativeAI/blob/main/5_01_Random_Forest_and_GBDT.ipynb).

**Objective:** predict credit risk using logistic regression, random forest, gradient boosting and XGBoost. Compare untuned and cross-validated tuned models; report performance on an untouched test set. The lecturer's key question is whether apparently perfect training scores imply good generalisation.

**Dataset:** 1,000 observations, 28 features, target `class` (0 = good borrower, 1 = bad borrower). The source is supplied separately through my.wbs. The dataset is deliberately **not included** in this public repository; upload the provided `credit_data_final.csv` when prompted in Google Colab. If your download is an Apple Numbers document, first export it as CSV.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Google Colab: upload credit_data_final.csv once, if it is not already present.
filename = Path('credit_data_final.csv')
if not filename.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError:
        raise FileNotFoundError('Place credit_data_final.csv beside this notebook, or run in Colab and upload it.')

assert filename.exists(), 'Upload the file with the exact name credit_data_final.csv.'
df = pd.read_csv(filename)
assert 'class' in df.columns, 'Expected target column: class'
assert df.shape == (1000, 29), f'Unexpected data shape: {df.shape}'
assert not df.isna().any().any(), 'Dataset contains missing values; investigate before training.'
assert set(df['class'].unique()) == {0, 1}, 'Expected binary 0/1 labels.'
print('Dataset:', df.shape)
print('Class distribution:', df['class'].value_counts().sort_index().to_dict())
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='class')
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=4567, stratify=y
)
print('Training:', X_train.shape, 'Testing:', X_test.shape)
print('Test class counts:', y_test.value_counts().sort_index().to_dict())
# All model selection and tuning below uses training data only.

## 1. Baseline models

A random forest trains many trees on different bootstrapped samples and combines their votes (**bagging**). GBDT and XGBoost build trees sequentially to correct prior errors (**boosting**). Logistic regression is the simpler benchmark.

Evaluate training and test scores separately: a perfect training score on its own is **not** evidence of perfect predictions for new borrowers.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'GBDT': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, n_jobs=2, eval_metric='logloss'),
}

def scores(y_true, y_pred):
    return {
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

baseline_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    for split, X_part, y_part in [('train', X_train, y_train), ('test', X_test, y_test)]:
        baseline_rows.append({'model': name, 'split': split, **scores(y_part, model.predict(X_part))})
baselines = pd.DataFrame(baseline_rows)
print(baselines.round(3).to_string(index=False))

## 2. Hyperparameter optimisation with five-fold cross-validation

Hyperparameters control learning before fitting (e.g. maximum tree depth, number of trees, learning rate). Search on the **training** split only using five-fold cross-validation. We keep the lecturer's `precision_macro` selection metric, but compare recall and F1 before choosing a practical credit-risk model.

Search ranges are kept plausible and reproducible. In particular, extremely high boosting learning rates in the original example can destabilise training. `n_iter=12` keeps the exercise reasonably fast to rerun.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

parameter_spaces = {
    'Logistic Regression': {
        'C': uniform(0.1, 19.9), 'penalty': ['l1', 'l2']
    },
    'Random Forest': {
        'n_estimators': randint(50, 350), 'max_depth': randint(2, 9),
        'min_samples_split': randint(2, 8), 'max_features': ['sqrt', 'log2', None]
    },
    'GBDT': {
        'n_estimators': randint(25, 200), 'learning_rate': uniform(0.01, 0.30),
        'max_depth': randint(1, 5)
    },
    'XGBoost': {
        'n_estimators': randint(25, 200), 'learning_rate': uniform(0.01, 0.30),
        'max_depth': randint(1, 6), 'subsample': uniform(0.7, 0.3)
    },
}

searches = {}
for name, model in models.items():
    search = RandomizedSearchCV(
        model, parameter_spaces[name], n_iter=12, cv=5,
        scoring='precision_macro', random_state=2015,
        n_jobs=-1, refit=True
    )
    search.fit(X_train, y_train)
    searches[name] = search
    print(f'{name}: CV macro precision {search.best_score_:.3f}; params: {search.best_params_}')

## 3. Final evaluation on the untouched test set

**Important:** the test set must not be used to optimise the parameters. It is used once here to assess generalisation. Macro precision treats the two classes equally, but can hide models that almost never identify a risky borrower.

In [ ]:
from sklearn.metrics import confusion_matrix

final_rows = []
for name, search in searches.items():
    pred = search.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    final_rows.append({
        'model': name, **scores(y_test, pred),
        'bad_borrowers_missed': int(cm[1, 0]),
        'bad_borrowers_found': int(cm[1, 1]),
        'predicted_bad': int((pred == 1).sum())
    })
results = pd.DataFrame(final_rows).sort_values('macro_precision', ascending=False)
print(results.round(3).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (name, search) in zip(axes, searches.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, search.predict(X_test), labels=[0, 1],
        display_labels=['Good (0)', 'Bad (1)'], ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 4. Interpretation and answers

**What is the difference between random forest and boosting?** Random forest builds many trees independently and averages their predictions, reducing variance. Gradient boosting sequentially adds trees to address remaining mistakes; learning rate and tree complexity affect overfitting.

**Why do random forest and XGBoost sometimes attain 100% on training data?** Highly flexible trees can memorise training observations. That is not a genuine 100% success rate on unseen borrowers. Test performance is the relevant check.

**What did tuning change?** Five-fold random search selects parameters using only training folds. Depending on the objective, it may sacrifice sensitivity to bad borrowers to increase macro precision. It does not guarantee test improvement.

**Which model is best?** It depends on error costs. A loan incorrectly approved for a risky borrower (true class 1, predicted class 0) is especially costly. The model with the highest macro precision can still be undesirable if it misses most bad borrowers. Use the confusion matrices together with recall and F1, and set the decision objective based on the cost of these mistakes.

**Verified run with the supplied dataset** (scikit-learn/XGBoost installed in this environment; numbers may vary across library versions):

| Tuned model | Test macro precision | Test macro recall | Test macro F1 | Risky borrowers missed (out of 60) |
|---|---:|---:|---:|---:|
| Logistic regression | 0.638 | 0.588 | 0.591 | 43 |
| Random forest | 0.855 | 0.525 | 0.463 | 57 |
| GBDT | 0.679 | 0.639 | 0.649 | 36 |
| XGBoost | 0.757 | 0.681 | 0.698 | 34 |

**Conclusion:** random forest leads macro precision but identifies only 3 of 60 bad borrowers. XGBoost has the best macro F1 in this run and misses fewer bad borrowers. It is the stronger candidate for a balanced classifier, although a lending decision would require cost-sensitive validation and additional checks. This is an educational exercise, not a production lending system.